# Brain Tumor MRI Classification — Deep Learning Practice

This notebook walks through dataset inspection, transfer-learning model setup, training, held-out evaluation, and prediction examples. It uses the same reusable modules as `train.py`.

**Safety note:** this is an educational classification exercise, not a medical device or a diagnostic tool.

Before running, download the public MRI image dataset linked in the README and arrange it under `data/Training/` and `data/Testing/`.


## 1. Imports and reproducibility

Run the notebook from the repository root so the `src/` package can be imported.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.brain_tumor_dl.data import count_images_by_class, load_datasets
from src.brain_tumor_dl.model import build_model
from src.brain_tumor_dl.evaluation import save_training_curves, evaluate_and_visualize

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
DATA_DIR = Path('data')
ARTIFACTS_DIR = Path('artifacts')
REPORTS_DIR = Path('reports')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)


## 2. Inspect the dataset

Counts are read from the local folders. This does not verify medical labels or image quality.

In [ ]:
train_counts = count_images_by_class(DATA_DIR / 'Training')
test_counts = count_images_by_class(DATA_DIR / 'Testing')
print('Training images:', train_counts)
print('Testing images: ', test_counts)
if not train_counts or not test_counts:
    raise FileNotFoundError(
        'Dataset not found. Read README.md and place Training/ and Testing/ inside data/.'
    )
if set(train_counts) != set(test_counts):
    raise ValueError('Training and Testing class folders must match exactly.')

class_count_frame = pd.DataFrame({
    'Training': pd.Series(train_counts),
    'Testing': pd.Series(test_counts),
}).fillna(0).astype(int)
ax = class_count_frame.plot(kind='bar', figsize=(9, 4), title='Image count per class')
ax.set_xlabel('Dataset class')
ax.set_ylabel('Number of images')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()


## 3. Load train, validation, and held-out test datasets

The validation subset is sampled from `Training/`. `Testing/` is kept separate until the final evaluation.

In [ ]:
train_ds, val_ds, test_ds, class_names = load_datasets(
    DATA_DIR,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    seed=SEED,
)
print('Class order:', class_names)


## 4. Visualize example images

The labels shown below are the directory labels in the public dataset, not a radiologist's independent review.

In [ ]:
images, labels = next(iter(train_ds))
count = min(12, len(images))
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for index, ax in enumerate(axes.flat):
    if index >= count:
        ax.axis('off')
        continue
    ax.imshow(np.clip(images[index].numpy(), 0, 255).astype(np.uint8))
    ax.set_title(class_names[int(labels[index].numpy())])
    ax.axis('off')
fig.suptitle('Training images and their folder labels')
plt.tight_layout()
plt.show()


## 5. Build the MobileNetV2 classifier

The feature extractor uses ImageNet-pretrained weights by default. The first run may download weights from Keras. Use `weights=None` below to initialize randomly if you cannot download them, understanding that the result will differ.

In [ ]:
model = build_model(
    num_classes=len(class_names),
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    weights='imagenet',  # change to None for random initialization
)
model.summary()


## 6. Train with validation monitoring

Early stopping restores the weights from the best validation-loss epoch. Do not choose model settings based on the final test scores.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / 'best_model.keras'),
        monitor='val_loss', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(str(REPORTS_DIR / 'training_history.csv')),
]
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks
)


## 7. Save the model and inspect learning curves

In [ ]:
model_path = ARTIFACTS_DIR / 'brain_tumor_mobilenetv2.keras'
model.save(model_path)
(ARTIFACTS_DIR / 'class_names.json').write_text(
    json.dumps(class_names, indent=2), encoding='utf-8'
)
save_training_curves(history, REPORTS_DIR)
print('Saved model:', model_path)
from IPython.display import Image as DisplayImage, display
display(DisplayImage(filename=str(REPORTS_DIR / 'training_curves.png')))


## 8. Evaluate on `Testing/` and generate evaluation visuals

The following computes scores only after training is complete. Record the result from your own run; do not assume an expected accuracy in advance.

In [ ]:
metrics = evaluate_and_visualize(
    model, test_ds, class_names, REPORTS_DIR, ARTIFACTS_DIR
)
print(json.dumps(metrics, indent=2))
for filename in ['confusion_matrix.png', 'class_metrics.png', 'prediction_examples.png']:
    path = REPORTS_DIR / filename
    if path.exists():
        print(filename)
        display(DisplayImage(filename=str(path)))


## 9. Try a single image

Set `IMAGE_PATH` to an image path within the local test dataset. The model's output is not clinical confidence.

In [ ]:
from PIL import Image

IMAGE_PATH = None  # Example: Path('data/Testing/glioma/your-image.jpg')
if IMAGE_PATH is not None:
    image = Image.open(IMAGE_PATH).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    pixels = np.asarray(image, dtype=np.float32)[None, ...]
    scores = model.predict(pixels, verbose=0)[0]
    ranking = np.argsort(scores)[::-1]
    display(image)
    for index in ranking:
        print(f'{class_names[int(index)]:15s} {float(scores[index]):.2%}')
else:
    print('Set IMAGE_PATH to a local image and rerun this cell.')


## Interpretation checklist

- Inspect the confusion matrix and per-class recall/F1; overall accuracy alone can hide failures.
- Mention the held-out test results honestly, using your own generated files in `artifacts/` and `reports/`.
- The public image dataset may contain biases and duplicates; a good score does not establish real-world generalization.
- This code classifies whole images. It does not segment or localize tumor tissue and it is not intended for patient care.
